# Embedding-Based Dictionary Expansion for Innovation

This Colab notebook demonstrates embedding-based dictionary expansion for finance/accounting NLP.

The exercise is inspired by Li, Mai, Shen, and Yan (2021, RFS), **"Measuring Corporate Culture Using Machine Learning"**, but this is a teaching demo rather than a full replication.

The notebook uses only two raw input CSV files from the GitHub `raw` branch:

- `003_all_US_calls_2024Q4_top500.csv`
- `003_all_US_calls_2024Q4_other.csv`

It does **not** depend on saved bigram/trigram files, saved Word2Vec models, or any local intermediate artifacts.

## 1. Setup

Install and import the packages needed for the demo.

If running in Colab, use **Runtime → Change runtime type → CPU**. GPU is not needed for Word2Vec.

In [ ]:
!pip -q install "numpy<2" "scipy<1.13" pandas gensim tqdm matplotlib

In [ ]:
import os
import re
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from gensim.models import Word2Vec
from gensim.models.phrases import Phrases, Phraser
import gensim.downloader as api

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_colwidth", 120)

## 2. Load Raw Q&A Data from GitHub

The full 2024Q4 US-company Q&A file is split into two GitHub-sized CSV files in the repository `raw` branch. We load both raw files and append them before doing any NLP.

In [ ]:
BASE_RAW_URL = "https://raw.githubusercontent.com/helenlu-vbs/NLP_LLM_for_Finance_and-Accounting_Research-Sheffield-/raw"
RAW_FILES = [
    "003_all_US_calls_2024Q4_top500.csv",
    "003_all_US_calls_2024Q4_other.csv",
]

df_parts = []
for file_name in RAW_FILES:
    url = f"{BASE_RAW_URL}/{file_name}"
    print("Loading:", url)
    part = pd.read_csv(url)
    part["source_file"] = file_name
    df_parts.append(part)

df = pd.concat(df_parts, ignore_index=True)
print("Combined shape:", df.shape)
print("Columns:", list(df.columns))
print("Unique firms:", df["companyid"].nunique())
print("Unique transcripts:", df["transcriptid"].nunique())
df.head()

## 3. Lightweight Cleaning for Teaching

The full Li et al. method uses a richer NLP pipeline: lemmatisation, NER replacement, dependency parsing, compound detection, stopword removal, phrase detection, and then Word2Vec.

For this classroom demo, we use a lightweight Word2Vec-style tokenizer:

- lowercases text
- lightly expands contractions
- keeps alphabetic and selected hyphenated terms such as `cost-effective`
- removes one-letter tokens and common stopwords
- keeps finance-relevant words such as `may`, `growth`, `margin`, `cash`, `risk`

These are **Word2Vec tokens**: words and phrase tokens. They are not LLM subword tokens.

In [ ]:
BASE_STOPWORDS = {
    "a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are",
    "as", "at", "be", "because", "been", "before", "being", "below", "between", "both", "but",
    "by", "can", "did", "do", "does", "doing", "down", "during", "each", "few", "for", "from",
    "further", "had", "has", "have", "having", "he", "her", "here", "hers", "herself", "him",
    "himself", "his", "how", "i", "if", "in", "into", "is", "it", "its", "itself", "just", "me",
    "more", "most", "my", "myself", "no", "nor", "not", "now", "of", "off", "on", "once", "only",
    "or", "other", "our", "ours", "ourselves", "out", "over", "own", "same", "she", "so", "some",
    "such", "than", "that", "the", "their", "theirs", "them", "themselves", "then", "there", "these",
    "they", "this", "those", "through", "to", "too", "under", "until", "up", "very", "was", "we",
    "were", "what", "when", "where", "which", "while", "who", "whom", "why", "will", "with", "you",
    "your", "yours", "yourself", "yourselves"
}

FINANCE_KEEPWORDS = {
    "may", "might", "must", "should", "growth", "margin", "cash", "flow", "revenue", "earnings",
    "cost", "costs", "demand", "supply", "risk", "capital", "investment", "market"
}
STOPWORDS = BASE_STOPWORDS.difference(FINANCE_KEEPWORDS)

TOKEN_RE = re.compile(r"[a-z]+(?:-[a-z]+)?")

def clean_and_tokenize(text):
    text = str(text).lower()
    replacements = {
        "can't": "cannot",
        "won't": "will not",
        "n't": " not",
        "'re": " are",
        "'ve": " have",
        "'ll": " will",
        "'d": " would",
        "'m": " am",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    tokens = []
    for token in TOKEN_RE.findall(text):
        if len(token) <= 1:
            continue
        if token in STOPWORDS:
            continue
        tokens.append(token)
    return tokens

text_col = "turn_text"
df = df[df[text_col].astype(str).str.strip().str.len() >= 20].copy()

sentences_unigram = [clean_and_tokenize(x) for x in tqdm(df[text_col], desc="Tokenizing Q&A turns")]
print("Rows after short-text filter:", len(sentences_unigram))
print("Example tokens:")
print(sentences_unigram[0][:80])

## 4. Phrase Detection

We use Gensim phrase detection to create phrase-aware tokens such as:

- `customer_experience`
- `operational_excellence`
- `artificial_intelligence`
- `gross_margin`

This notebook trains bigram and trigram phrasers from scratch each time. It does not load saved phrase files.

In [ ]:
phrase_min_count = 10
phrase_threshold = 10

t0 = time.time()
bigram_phrases = Phrases(sentences_unigram, min_count=phrase_min_count, threshold=phrase_threshold)
bigram = Phraser(bigram_phrases)
sentences_bigram = [bigram[s] for s in sentences_unigram]

trigram_phrases = Phrases(sentences_bigram, min_count=phrase_min_count, threshold=phrase_threshold)
trigram = Phraser(trigram_phrases)
sentences_trigram = [trigram[s] for s in sentences_bigram]

print(f"Phrase detection time: {time.time() - t0:.1f} seconds")
print("Example phrase-aware tokens:")
print(sentences_trigram[0][:80])

## 5. Train Word2Vec on Earnings-Call Q&A Text

We train a skip-gram Word2Vec model on the phrase-aware corpus.

Settings:

- `vector_size=50`
- `window=5`
- `min_count=10`
- `sg=1`
- `negative=5`
- `epochs=3`

In [ ]:
w2v_params = dict(
    vector_size=50,
    window=5,
    min_count=10,
    sg=1,
    negative=5,
    epochs=3,
    workers=2,
    seed=SEED,
)

t0 = time.time()
w2v_model = Word2Vec(sentences=sentences_trigram, **w2v_params)
training_time = time.time() - t0

total_tokens = sum(len(s) for s in sentences_trigram)
print("Sentences/turns:", f"{len(sentences_trigram):,}")
print("Total tokens:", f"{total_tokens:,}")
print("Vocabulary size:", f"{len(w2v_model.wv):,}")
print("Training time:", f"{training_time:.1f} seconds")

## 6. Innovation Seed Words

We use a broader Li-style innovation seed list. This gives the local Word2Vec model a stronger anchor than a narrow seed list.

In [ ]:
li_innovation_seeds = [
    "innovation", "innovate", "innovative",
    "creativity", "creative", "create",
    "passion", "passionate",
    "efficiency", "efficient",
    "excellence", "pride",
]

covered_w2v = [w for w in li_innovation_seeds if w in w2v_model.wv]
missing_w2v = [w for w in li_innovation_seeds if w not in w2v_model.wv]
print("Covered Word2Vec seeds:", covered_w2v)
print("Missing Word2Vec seeds:", missing_w2v)

## 7. Nearest Neighbours from Word2Vec

We average the vectors for covered seed words, compute cosine similarity to every vocabulary term, and return the top 20 nearest neighbours.

In [ ]:
def nearest_to_seed_average(kv, seeds, topn, source):
    covered = [w for w in seeds if w in kv]
    seed_set = set(covered)
    avg_vec = np.mean([kv[w] for w in covered], axis=0)
    avg_vec = avg_vec / np.linalg.norm(avg_vec)

    matrix = kv.vectors
    matrix_norm = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    sims = matrix_norm @ avg_vec

    rows = []
    for idx in np.argsort(-sims):
        word = kv.index_to_key[int(idx)]
        if word in seed_set:
            continue
        rows.append({
            "rank": len(rows) + 1,
            "word": word,
            "similarity": float(sims[int(idx)]),
            "source": source,
        })
        if len(rows) >= topn:
            break
    return pd.DataFrame(rows)

w2v_top20 = nearest_to_seed_average(
    w2v_model.wv,
    li_innovation_seeds,
    topn=20,
    source="word2vec_2024q4_qna_split_full",
)

w2v_top20

## 8. Compare with GloVe

GloVe is trained on broad general-English data. It often returns semantically clean general-English neighbours, while the local Word2Vec model learns the language of earnings calls.

In [ ]:
try:
    glove = api.load("glove-wiki-gigaword-100")
    covered_glove = [w for w in li_innovation_seeds if w in glove]
    missing_glove = [w for w in li_innovation_seeds if w not in glove]
    print("Covered GloVe seeds:", covered_glove)
    print("Missing GloVe seeds:", missing_glove)
    glove_top20 = nearest_to_seed_average(
        glove,
        li_innovation_seeds,
        topn=20,
        source="glove_wiki_gigaword_100",
    )
    display(glove_top20)
except Exception as e:
    glove_top20 = None
    print("GloVe download failed. This section can be skipped in class or run when internet is available.")
    print(e)

## 9. Compare with Li et al.'s Final Innovation Dictionary

The list below is the final innovation dictionary supplied for this teaching exercise. We compare the top-20 Word2Vec and GloVe neighbours with this dictionary.

Two matching rules:

1. **Exact match**: candidate appears exactly in the dictionary.
2. **Normalized match**: hyphens and underscores are treated as equivalent, e.g. `cost-effective` ≈ `cost_effective`.

In [ ]:
li_final_dictionary_text = """
creativity, innovative, innovate, innovation, creative, excellence, passion, world-class,
technology, operational_excellence, passionate, product_innovation, capability,
customer_experience, thought_leadership, expertise, agility, efficient,
technology_innovation, competency, know-how, cutting-edge, agile, creatively,
customer-centric, enable, value_proposition, reinvent, focus, innovation_capability,
efficiency, customer_value, customer_intimacy, competence, user_experience, create,
storytelling, pride, core_competency, ingenuity, technology_platform,
consumer_experience, product_technology, engineering_team, differentiate, powerful,
inspiring, innovation_process, transform, product_team, inspiration, innovation_team,
technology_team, best-in-class, r&d_team, loyalty, truly, technological,
differentiation, technology_capability, intellect, focused, design_capability,
product_development, solve_customer_problem, customer_focus, inspire, branding,
cut_edge, business_process, brand, personalization, distinctive, cost-effective,
automation, world_class, harness, efficiently, domain_expertise,
product_development_capability, cost-efficient, core_capability, consumer_insight,
platform, engaging, delight, mass_customization, uniqueness, product_leadership,
customer_success, specialization, innovation_engine, invent, guest_experience,
innovator, tool, design_team, craftsmanship, seamlessness, intellectual_property,
solve_problem, incredible, go-to-market, service_experience, enhance,
technology_standpoint, sophistication, excitement, innovatively, great,
business_model, world-leading, innovation_lab, fanatical_support,
brand_management, service_model, go-to-market_capability, customer_insight,
authentic, discipline, nimble, effectiveness, customer-oriented, design_thinking,
execution, mobile-first, knowhow, product_idea, relentless, r&d_capability,
importantly, product_development_team, customer-focused, product_design,
showcase, innovation_standpoint, core_competence, ai_technology, excel, develop,
effort, responsiveness, process_excellence, building_capability, technology_solution,
product_capability, execution_capability, critically_important, solution, heritage,
simplicity, cohesive, scalability, intelligent, curation, process_improvement, intimacy,
user_interface, r&d_organization, best-in-breed, core_technology, analytic,
domain_knowledge, creativeness, client_experience, technology_perspective,
invention, cost_efficiency, technologically, core_strength, award-winning, learn,
merchandising, marketing_team, ethos, optimize, awareness, technology_leadership,
game_team, leadership_position, engineering_capability, leverage_technology,
feature_functionality, brand_equity, smarter, enabler, dna, operating_platform,
computer_graphic, service_excellence, marketing_idea, service_delivery_platform,
artistic, product_development_process, ability, reimagine, platform_capability,
democratize, end-to-end, forefront, connectedness, customer_interface,
datum_analytic, innovation_perspective, r&d_department, take_cost_out, reengineer,
workflow, center_excellence, marketing_technology, relevancy, unparalleled, content,
successful, smart, technology_architecture, process_innovation, authenticity, scalable,
vision, marketer, visual_merchandising, brand_experience, productivity, technologyenabled,
terrific, easy-to-use, product_experience, coherence, product_management,
machine_learning_ai, leadership_product, industry_leadership, simplify, science,
versatility, artificial_intelligence, packaging_solution, intellectual, datum_science,
best-of-breed, attract, adaptability, r&d_group, drive_innovation, delivery_platform,
succeed, modern, state-of-the-art, immersive, information_technology,
engineering_skill, r&d_community, transformation, ease-of-use, design,
category_management, technology_base, business_system, unique,
application_expertise, video_technology, product_creation, breakthrough_technology,
teaching, innovation_technology, delivery_system, breadth_depth,
marketing_capability, visual, world_class_product, technology-driven,
internally_externally, delivery_model, consumer_engagement, success,
rapid_prototyping, customer_centricity, information-based, problem_solver,
delivery_organization, video_experience, globalize, product_excellence, problemsolving,
machine_learning, product_offering, marketing_expertise, social_media,
customer_loyalty, design_expertise, personalized, unique_selling_proposition,
marketing_skill, enablement, product_developer, service_leader,
engineering_organization, usability, technology_development,
manufacturing_engineering, innovativeness, leadership_model,
technology_organization, entertainment_experience, imaginative,
product_differentiation, resourceful, search_capability, consumer-centric, creator,
brand_recognition, shopping_experience, innovation_center,
breakthrough_innovation, knowledge-based, design_standpoint,
knowledge_management, content_creation, secret_sauce, core_business_process,
multi-channel, software_team, software_engineering, distinctiveness,
store_environment, imperative, compelling, globalization,
customer_relationship_management, product_development_system,
core_value_proposition, product_functionality, operation_excellence, prowess,
resonate, fabulous, technology-based, process_management, newness, exciting, clever,
restaurant_experience, recipe, marketing_tool, supply_chain_approach,
technology_differentiation, proven, storyteller, devops, inventive, architect,
product_solution, deep_domain_expertise, technology_leader, engineering_expertise,
amazing, solution_capability, engineering_talent, innovation_side,
application_knowledge, consumer_understanding, experiential,
solve_business_problem, fantastic, brand_name, service_culture, brand_building,
search_technology, testament, unifying, organizations, workspace, foundation,
brand_identity, inventiveness, brand_positioning, integrated, wonderful, fanatical,
best, messaging, mastery, fun, self-expression, store_experience, first-rate, elegance,
marketing_excellence, content_experience, beautiful, consulting_expertise,
operating_skill, brain_power, taste, inspirational, hallmark, superb
"""

li_terms = []
for item in re.split(r",|\n", li_final_dictionary_text):
    term = item.strip().lower()
    if term:
        li_terms.append(term)
li_terms = list(dict.fromkeys(li_terms))

li_exact = set(li_terms)
li_normalized = {x.replace("-", "_") for x in li_terms}
print("Number of unique Li final innovation dictionary terms:", len(li_terms))

In [ ]:
def compare_to_li_dictionary(df_candidates, method_name):
    words = df_candidates["word"].str.lower().tolist()
    exact_hits = [w for w in words if w in li_exact]
    normalized_hits = [w for w in words if w.replace("-", "_") in li_normalized]
    return {
        "method": method_name,
        "top_n": len(words),
        "exact_hits": len(exact_hits),
        "exact_hit_rate": len(exact_hits) / len(words),
        "exact_hit_words": "; ".join(exact_hits),
        "normalized_hits": len(normalized_hits),
        "normalized_hit_rate": len(normalized_hits) / len(words),
        "normalized_hit_words": "; ".join(normalized_hits),
    }

comparison_rows = [compare_to_li_dictionary(w2v_top20, "Word2Vec on 2024Q4 Q&A")]
if glove_top20 is not None:
    comparison_rows.append(compare_to_li_dictionary(glove_top20, "GloVe wiki-gigaword-100"))

comparison = pd.DataFrame(comparison_rows)
comparison

## 10. Teaching Interpretation

In this run, GloVe may have a higher raw hit count against the final Li innovation dictionary because it knows many general-English dictionary words such as `focus`, `ability`, `awareness`, `enhance`, `unique`, or `artistic`.

The earnings-call Word2Vec model often returns more domain-specific innovation candidates such as `operational_excellence`, `artificial_intelligence`, `scalable`, `cost-effective`, `digital_transformation`, and `tech_stack`.

This is the key teaching point:

- **GloVe** is strong for general semantic similarity.
- **Word2Vec trained on earnings calls** is useful for domain-specific business language.
- The embedding output is a candidate list, not the final dictionary.
- Researchers must inspect, clean, validate, and document dictionary choices.
- A one-quarter corpus is enough for classroom demonstration, not for a publishable dictionary.

## 11. Optional Simple Scoring Exercise

Students can choose accepted innovation terms after inspecting the candidates, then score each Q&A turn by counting accepted terms divided by total tokens.

In [ ]:
accepted_innovation_words = {
    "innovation", "innovative", "creative", "create", "efficiency", "efficient",
    "operational_excellence", "artificial_intelligence", "scalable", "cost-effective",
    "customer_experience", "value_proposition", "automation", "platform"
}
accepted_normalized = {w.replace("-", "_") for w in accepted_innovation_words}

def score_turn(text):
    tokens = clean_and_tokenize(text)
    phrase_tokens = trigram[bigram[tokens]]
    normalized = [t.replace("-", "_") for t in phrase_tokens]
    hits = sum(1 for t in normalized if t in accepted_normalized)
    total = len(normalized)
    return hits, total, hits / total if total else 0.0

scores = df.copy()
triples = scores[text_col].map(score_turn)
scores["innovation_word_hits"] = [x[0] for x in triples]
scores["token_count"] = [x[1] for x in triples]
scores["innovation_score"] = [x[2] for x in triples]

firm_scores = (
    scores.groupby(["companyname", "companyid", "speaker_type"], dropna=False)
    .agg(
        turns=("innovation_score", "size"),
        innovation_word_hits=("innovation_word_hits", "sum"),
        token_count=("token_count", "sum"),
    )
    .reset_index()
)
firm_scores["innovation_score"] = firm_scores["innovation_word_hits"] / firm_scores["token_count"]
firm_scores.sort_values("innovation_score", ascending=False).head(20)